# 2. Generando Incrustaciones contextuales (Contextual embeddings)

## 2.1 Carga y transformacion de informacion

In [21]:
#!pip install openpyxl
#!pip install demoji
#!pip install unidecode
#!pip install sentence-transformers umap-learn plotly scikit-learn tensorflow

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
from unidecode import unidecode
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================================
# Rutas del proyecto
# ============================================================================
PROJECT_DIR = Path(r"D:\Datasets\atribucion-autoria-tweets")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Archivo de entrada
data_path = PROCESSED_DIR / "fullclases.csv"

# ============================================================================
# Carga de datos
# ============================================================================
merged_data = pd.read_csv(data_path)
merged_data.info(verbose=True, memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 963498 entries, 0 to 963497
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             963498 non-null  int64 
 1   tweetText      963498 non-null  object
 2   tweetURL       963498 non-null  object
 3   type           963498 non-null  object
 4   tweetAuthor    959858 non-null  object
 5   handle         963498 non-null  object
 6   replyCount     963498 non-null  int64 
 7   quoteCount     963498 non-null  int64 
 8   retweetCount   963498 non-null  int64 
 9   likeCount      963498 non-null  int64 
 10  views          963498 non-null  object
 11  bookmarkCount  963498 non-null  int64 
 12  createdAt      963498 non-null  object
 13  allMediaURL    250166 non-null  object
 14  videoURL       65276 non-null   object
 15  filename       963498 non-null  object
 16  Final_user     963498 non-null  object
dtypes: int64(6), object(11)
memory usage: 912.8 MB


In [16]:
merged_data.head()

,id,tweetText,tweetURL,type,tweetAuthor,handle,replyCount,quoteCount,retweetCount,likeCount,views,bookmarkCount,createdAt,allMediaURL,videoURL,filename,Final_user
0,1894501879817584955,Pues resulta que el ya famoso despacho morenis...,https://x.com/AccionNacional/status/1894501879...,retweet,Elías Lixa,@eLiasLixa,0,0,2391,0,-,0,2025-02-25 15:37:00,NaN,NaN,TwExtract-AccionNacional-20250226_082459,AccionNacional
1,1894501607640842641,¡Las mujeres y niñas merecen vivir libres de v...,https://x.com/AccionNacional/status/1894501607...,tweet,Acción Nacional,@AccionNacional,3,0,3,3,654,1,2025-02-25 15:35:56,https://pbs.twimg.com/media/GkqfK5VXoAEeuCD.jpg,NaN,TwExtract-AccionNacional-20250226_082459,AccionNacional
2,1894476017907241089,Sostuvimos una productiva reunión de trabajo e...,https://x.com/AccionNacional/status/1894476017...,retweet,Senadores del PAN,@SenadoresdelPAN,0,0,8,0,-,0,2025-02-25 13:54:14,NaN,NaN,TwExtract-AccionNacional-20250226_082459,AccionNacional
3,1894459390746661291,ENTREGA DE LA OBRA DE AMPLIACIÓN DEL PUENTE DE...,https://x.com/AccionNacional/status/1894459390...,retweet,Tere Jiménez,@TereJimenezE,0,0,5,0,-,0,2025-02-25 12:48:10,NaN,NaN,TwExtract-AccionNacional-20250226_082459,AccionNacional
4,1894456464963768747,"¡Muy feliz cumpleaños Senador @RicardoAnayaC, ...",https://x.com/AccionNacional/status/1894456464...,tweet,Acción Nacional,@AccionNacional,117,7,75,426,6800,3,2025-02-25 12:36:33,https://pbs.twimg.com/media/Gkp2HLFW8AAwz5K.jpg,NaN,TwExtract-AccionNacional-20250226_082459,AccionNacional


In [17]:
# List of users to remove
users_to_remove = ['DanielNoboaOk','DiegoFC','jorgeramosnews','JorgeGarcesMx', 'RobertoMadrazo_','LuisitoComunica','RicardoAnayaC', 'AlfredoJalife','PagesBeatriz', 'EstefaniaVeloz']  # Add your users here

# Remove the specified users using query()
merged_data = merged_data.query("Final_user not in @users_to_remove")
merged_data.info(verbose=True, memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 928759 entries, 0 to 963497
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             928759 non-null  int64 
 1   tweetText      928759 non-null  object
 2   tweetURL       928759 non-null  object
 3   type           928759 non-null  object
 4   tweetAuthor    928759 non-null  object
 5   handle         928759 non-null  object
 6   replyCount     928759 non-null  int64 
 7   quoteCount     928759 non-null  int64 
 8   retweetCount   928759 non-null  int64 
 9   likeCount      928759 non-null  int64 
 10  views          928759 non-null  object
 11  bookmarkCount  928759 non-null  int64 
 12  createdAt      928759 non-null  object
 13  allMediaURL    239212 non-null  object
 14  videoURL       63109 non-null   object
 15  filename       928759 non-null  object
 16  Final_user     928759 non-null  object
dtypes: int64(6), object(11)
memory usage: 887.4 MB


In [18]:
# Extract unique names from the 'Nombre_Personaje' column
unique_names = merged_data['Final_user'].unique()

# Create the dictionary mapping names to indices starting from 1 to 37
diccionario_nombres = dict(zip(unique_names, range(0, 60)))  # Changed range to (1, 38)
diccionario_nombres

{'AccionNacional': 0,
 'AgustinLaje': 1,
 'alitomorenoc': 2,
 'beltrandelrio': 3,
 'brozoxmiswebs': 4,
 'carlaescoffie': 5,
 'carlosalazraki': 6,
 'cesarpinedar': 7,
 'CFKArgentina': 8,
 'ChumelTorres': 9,
 'ClaudioXGG': 10,
 'DeniseDresserG': 11,
 'DiazCanelB': 12,
 'EdySmol': 13,
 'EVerastegui': 14,
 'fede_bonasso': 15,
 'FelipeCalderon': 16,
 'fernandeznorona': 17,
 'GabrielBoric': 18,
 'GlodeJo07': 19,
 'GloriaAlvarez85': 20,
 'HernanGomezB': 21,
 'jet1403': 22,
 'Jjlopez_almejo': 23,
 'JLMNoticias': 24,
 'JMilei': 25,
 'LAURAZAPATAM': 26,
 'LeonKrauze': 27,
 'LillyTellez': 28,
 'lopezobrador_': 29,
 'MariaCorinaYA': 30,
 'MarkoCortes': 31,
 'MovCiudadanoMX': 32,
 'nayibbukele': 33,
 'NicolasMaduro': 34,
 'PabloIglesias': 35,
 'PartidoMorenaMx': 36,
 'partidoverdemex': 37,
 'petrogustavo': 38,
 'PonchoGutz': 39,
 'RevistaMigala': 40,
 'RicardoBSalinas': 41,
 'sanchezcastejon': 42,
 'VicenteFoxQue': 43,
 'Viri_Rios': 44,
 'XochitlGalvez': 45,
 'YosoyPedrero': 46}

In [19]:
# perform mapping
merged_data['Tweetero_num'] = merged_data['Final_user'].map(diccionario_nombres)

# Keep only needed columns
columns_to_keep = ['tweetText', 'Tweetero_num']
df_filtered = merged_data.drop(columns=[col for col in merged_data.columns if col not in columns_to_keep])

In [20]:
# remove NaN
df_cleaned = df_filtered.dropna(subset=['tweetText'])

In [21]:
df_cleaned

,tweetText,Tweetero_num
0,Pues resulta que el ya famoso despacho morenis...,0
1,¡Las mujeres y niñas merecen vivir libres de v...,0
2,Sostuvimos una productiva reunión de trabajo e...,0
3,ENTREGA DE LA OBRA DE AMPLIACIÓN DEL PUENTE DE...,0
4,"¡Muy feliz cumpleaños Senador @RicardoAnayaC, ...",0
...,...,...
963493,"@Miriam_Junne Hasta aquí contigo, bye",46
963494,"Antes, @lopezobrador_ decía que si de que serv...",46
963495,Spoiler: El único que acabará con el machismo ...,46
963496,@KarlaRiveraMX Si,46


In [23]:
df_cleaned.to_csv(r'D:\Datasets\atribucion-autoria-tweets\data\processed\df_cleaned.csv', index=False)

## 2.2 Implementacion de Modelos

### 2.2.1. SentenceTransformer

In [12]:
import torch
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import umap
import plotly.express as px
import plotly.graph_objects as go
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Total de tweets: {len(df_cleaned)}")
print(f"Número de autores únicos: {df_cleaned['Tweetero_num'].nunique()}")

Total de tweets: 928759
Número de autores únicos: 47


In [14]:
# ============================================================================
# 3. GENERACIÓN DE EMBEDDINGS CON SENTENCE TRANSFORMER
# ============================================================================

print("Generando embeddings...")

model = SentenceTransformer('dccuchile/bert-base-spanish-wwm-uncased')

batch_size_textos = 5000       # número de tweets por bloque
batch_size_modelo = 64         # batch interno del modelo

all_embeddings = []

for i in range(0, len(df_cleaned), batch_size_textos):

    print(f"\nProcesando tweets {i:,} - {min(i+batch_size_textos, len(df_cleaned)):,}")

    textos = df_cleaned["tweetText"].iloc[i:i+batch_size_textos].tolist()

    emb = model.encode(
        textos,
        batch_size=batch_size_modelo,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    all_embeddings.append(emb)

embeddings = np.vstack(all_embeddings)

print(f"\nShape de embeddings: {embeddings.shape}")
print(f"Varianza total: {np.var(embeddings):.6f}")

Generando embeddings...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Procesando tweets 0 - 5,000


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

...


Procesando tweets 920,000 - 925,000


Batches:   0%|          | 0/79 [00:00<?, ?it/s]


Procesando tweets 925,000 - 928,759


Batches:   0%|          | 0/59 [00:00<?, ?it/s]


Shape de embeddings: (928759, 768)
Varianza total: 0.001297


In [23]:
# Etiquetas numéricas de los autores
y_encoded = df_cleaned["Tweetero_num"].to_numpy()

print(f"Embeddings: {embeddings.shape}")
print(f"Etiquetas: {y_encoded.shape}")
print(f"Número de clases: {len(np.unique(y_encoded))}")

Embeddings: (928759, 768)
Etiquetas: (928759,)
Número de clases: 47


In [24]:
from pathlib import Path

output_dir = Path(r"D:\Datasets\atribucion-autoria-tweets\data\embeddings\SentenceTransformer")

np.save(output_dir / "embeddings_sentence_transformer.npy", embeddings)
np.save(output_dir / "y_autores.npy", y_encoded)

print("Archivos guardados.")

Archivos guardados.


In [25]:
import joblib

joblib.dump(
    diccionario_nombres,
    output_dir / "diccionario_autores.pkl"
)

print("✅ Diccionario guardado")

✅ Diccionario guardado


In [26]:
emb = np.load(output_dir / "embeddings_sentence_transformer.npy")
y = np.load(output_dir / "y_autores.npy")

print("Embeddings:", emb.shape)
print("Etiquetas:", y.shape)

assert emb.shape == embeddings.shape
assert y.shape == y_encoded.shape

print("✅ Verificación exitosa")

Embeddings: (928759, 768)
Etiquetas: (928759,)
✅ Verificación exitosa


### 2.2.2. Mex_state

Este modelo basado en Roberta se entrenó usando más de 140 millones de tweets de México en español. Recolectados entre diciembre del 2015 y febrero del 2023. 

[Link](https://huggingface.co/guillermoruiz/mex_state)

#### 2.2.2.1. Usando batches y generando varios archivos

In [16]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np
import os

# ============================================================================
# 1. CARGAR MODELO
# ============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

tokenizer = AutoTokenizer.from_pretrained("guillermoruiz/mex_state")

model = AutoModelForMaskedLM.from_pretrained(
    "guillermoruiz/mex_state",
    output_hidden_states=True
)

model.to(device)
model.eval()

Dispositivo: cpu


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(30006, 512, padding_idx=1)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(512, 512, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): La

In [17]:
# ============================================================================
# 2. RUTA DE SALIDA
# ============================================================================
output_dir_mx = r"D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state"
os.makedirs(output_dir_mx, exist_ok=True)

In [18]:
# ============================================================================
# 3. FUNCIÓN PARA EMBEDDINGS
# ============================================================================
def get_embeddings(texts, batch_size=64):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):

        batch = ["[MASK] _GEO " + str(t) for t in texts[i:i+batch_size]]

        tokens = tokenizer(
            batch,
            return_tensors="pt",
            max_length=256,
            padding="max_length",
            truncation=True
        )

        tokens = {
            "input_ids": tokens["input_ids"].to(device),
            "attention_mask": tokens["attention_mask"].to(device)
        }

        with torch.no_grad():
            outputs = model(**tokens)

        emb = outputs.hidden_states[-1][:, 1]

        all_embeddings.append(emb.cpu())

    return torch.cat(all_embeddings)

In [19]:
chunk_size = 5000

In [20]:
# ============================================================================
# 4. GENERAR EMBEDDINGS
# ============================================================================

texts = df_cleaned["tweetText"].astype(str).tolist()

chunk_size = 5000

for start in range(0, len(texts), chunk_size):

    end = min(start + chunk_size, len(texts))

    print(f"\nProcesando {start:,} - {end:,}")

    emb = get_embeddings(
        texts[start:end],
        batch_size=64
    )

    filename = os.path.join(
        output_dir_mx,
        f"embeddings_{start}_{end}.pt"
    )

    torch.save(emb, filename)

    print(f"Guardado: {filename}")

    del emb

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Procesando 0 - 5,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_0_5000.pt

Procesando 5,000 - 10,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_5000_10000.pt

Procesando 10,000 - 15,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_10000_15000.pt

Procesando 15,000 - 20,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_15000_20000.pt

Procesando 20,000 - 25,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_20000_25000.pt

Procesando 25,000 - 30,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_25000_30000.pt

Procesando 30,000 - 35,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_30000_35000.pt

Procesando 35,000 - 40,000
Guardado: D:\Datasets\atribucion-autoria-tweets\data\embeddings\mex_state\embeddings_3500

#### 2.2.2.2. Batches en un solo archivo

In [8]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm import tqdm

# Configuración
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8  # Reduce según tu memoria RAM
CHUNK_SIZE = 10000  # Tamaño del chunk para guardar

print(f"Usando dispositivo: {DEVICE}")

# Cargar modelo y tokenizer
tokenizer = AutoTokenizer.from_pretrained("guillermoruiz/mex_state")
model = AutoModelForMaskedLM.from_pretrained(
    "guillermoruiz/mex_state", 
    output_hidden_states=True
)
model = model.to(DEVICE)
model.eval()

def process_tweets(text_list, bs=BATCH_SIZE, chunk_size=CHUNK_SIZE):
    """Procesa tweets en chunks con checkpointing"""
    
    total_chunks = (len(text_list) + chunk_size - 1) // chunk_size
    all_embeddings = []
    
    for chunk_idx in tqdm(range(total_chunks), desc="Procesando chunks"):
        start = chunk_idx * chunk_size
        end = min((chunk_idx + 1) * chunk_size, len(text_list))
        chunk_text = text_list[start:end]
        
        # Preparar textos para chunk
        chunk_text = ["[MASK] _GEO " + t for t in chunk_text]
        chunk_embeddings = []
        
        for i in range(0, len(chunk_text), bs):
            # Tokenizar
            tokens = tokenizer(
                chunk_text[i:i+bs],
                return_tensors="pt",
                max_length=256,
                padding="max_length",
                truncation=True
            )
            
            # Mover a dispositivo
            input_ids = tokens['input_ids'].to(DEVICE)
            attention_mask = tokens['attention_mask'].to(DEVICE)
            
            # Inferencia
            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            # Extraer embeddings
            embeddings = outputs.hidden_states[-1][:, 1].cpu()
            chunk_embeddings.append(embeddings)
            
            # Limpiar memoria
            del tokens, input_ids, attention_mask, outputs, embeddings
            torch.cuda.empty_cache()
        
        # Concatenar embeddings del chunk
        chunk_embeddings = torch.cat(chunk_embeddings)
        
        # Guardar chunk en disco
        chunk_file = f"embeddings_chunk_{chunk_idx}.pt"
        torch.save(chunk_embeddings, chunk_file)
        all_embeddings.append(chunk_file)
        
        # Limpiar memoria
        del chunk_embeddings
        gc.collect()
    
    return all_embeddings

# Ejecutar
text = df_cleaned['tweetText'].astype(str).tolist()
embedding_files = process_tweets(text)

print(f"Procesados {len(text)} tweets en {len(embedding_files)} archivos")

# Para cargar todos los embeddings después:
all_embeddings = []
for file in embedding_files:
    all_embeddings.append(torch.load(file))
final_embeddings = torch.cat(all_embeddings)

Usando dispositivo: cpu


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Procesando chunks: 100%|██████████████████████████████████████████████████████████| 93/93 [27:23:06<00:00, 1060.08s/it]


Procesados 928759 tweets en 93 archivos


##### Exportar objeto pytorch con los embeddings

In [10]:
torch.save(final_embeddings, 'D:/Datasets/atribucion-autoria-tweets/data/embeddings/mex_state_local/embeddings_47c_7000.pt') 

### 2.2.3. RoBERTuito

RoBERTuito es un modelo de lenguaje preentrenado para contenido generado por usuarios en español, entrenado siguiendo las directrices de RoBERTa con 500 millones de tweets. RoBERTuito está disponible en 3 versiones: con mayúsculas y minúsculas, sin mayúsculas y minúsculas, y sin acentos.

[Link](https://github.com/sullivansantana/robertuito/tree/main)

**Nota: Los embeddings de RoBERTuito se generaron externamente debido al tiempo de cómputo requerido. Se utilizan en el Notebook 03 y 04 para clasificación con red neuronal y las maquinas de Soporte vectorial.**